# Shopping-Mall Evacuation: How Exit Width Must Scale with Crowd Size

This notebook simulates the evacuation of a simplified single-floor shopping mall (about 40 m × 30 m, with rectangular shop blocks and two exits) using the **Social Force Model**. Each shopper is an agent with its own desired walking speed. A driving force pushes them towards the nearest exit, and repulsive and contact forces from other people, walls and shop blocks push them away. Pedestrians follow a precomputed shortest-distance map, so they route around the shops. Their equations of motion are integrated with **Velocity-Verlet**, and **cell lists** find nearby pedestrians efficiently. Every scenario is repeated over many random seeds and reported with **95 % confidence intervals**. Sweeping exit width at crowd sizes from 100 to 400 people gives the minimum width that clears the floor within a 3-minute target, and shows whether that width must grow faster than occupancy.

**Key result:** *[to be completed in Section 6 — required exit width $w^*(N)$ and scaling exponent $\alpha$ with 95 % CIs]*

**Verification:** a single pedestrian's speed is checked against the analytic relaxation curve $v(t)=v_0\left(1-e^{-t/\tau}\right)$. The notebook also checks that people are conserved at every step, that results converge as the timestep shrinks (dt, dt/2, dt/4) while a too-large step goes unstable, and that exit flow is plausible against published bottleneck measurements (Section 5).

**How to run:** *Runtime → Run all* on a fresh Google Colab runtime, or *Run All* in a local Jupyter install. Missing packages are installed automatically in the first code cell. All random seeds are fixed and stated.

In [ ]:
# Setup: install anything missing, then import.
# Works unchanged on a fresh Google Colab runtime and in a local Jupyter install.
import importlib.util, subprocess, sys

REQUIRED = ["numpy", "scipy", "matplotlib", "pandas", "numba"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import platform, time
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numba
from numba import njit

# Reproducibility: every random draw in this notebook comes from a seeded generator.
SEED = 42
rng = np.random.default_rng(SEED)

# Run mode: QUICK_MODE uses fewer replications so "Run all" stays under ~10 minutes;
# set to False to regenerate the full study (more seeds per scenario).
QUICK_MODE = True

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print(f"Python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"pandas {pd.__version__} | matplotlib {matplotlib.__version__} | numba {numba.__version__}")
print(f"SEED = {SEED} | QUICK_MODE = {QUICK_MODE}")

## 1. Problem and question

Large shopping malls in Colombo hold hundreds of shoppers on one open floor. In an emergency, everyone converges on a few exits at once, and the exits become bottlenecks: the floor can only empty as fast as people can pass through the doorways. If the exits are too narrow for the number of people inside, queues and crushing build up in front of them and evacuation takes much longer than the walking distance alone suggests.

**Question:** *How wide must the exits be to clear the floor within a target time of 3 minutes (180 s) as the number of occupants rises from 100 to 400?*

| Quantity | Symbol | Values studied |
|---|---|---|
| Number of occupants | $N$ | 100, 200, 300, 400 people |
| Width of each of the two exits | $w$ | swept over a range of widths (initially 1.0–3.0 m, refined by a pilot run in Section 6) |
| Evacuation time (last person out) | $T_\text{evac}$ | output, s, mean ± 95 % CI over seeds |
| Target evacuation time | $T_\text{target}$ | 180 s (assumed) |
| Required exit width | $w^*(N)$ | smallest $w$ with $T_\text{evac} \le T_\text{target}$ |

To see how the required width scales with occupancy, a power law $w^*(N) \propto N^{\alpha}$ is fitted. If $\alpha > 1$, exit width must grow **faster** than occupancy: doubling the number of people needs more than double the exit width. If $\alpha \approx 1$, width grows in proportion; if $\alpha < 1$, it grows more slowly.

**Why it matters:** exit width and the maximum safe occupancy are linked design decisions. Knowing how $w^*$ scales with $N$ shows whether an existing floor can safely hold more people, and how much wider the exits would need to be.

**Scope:** one floor with synthetic geometry. Model parameters come from the published Social Force Model literature. Everyone starts moving at $t = 0$ (no pre-movement delay). As a stretch scenario, one exit is blocked at the start of the emergency, and the increase in evacuation time and the peak crowd density at the remaining exit are measured.

## 2. Model, assumptions and equations

Full derivations and justifications are in `THEORY.md`. Reference numbers [n] refer to Section 7.

### 2.1 Geometry

The mall floor is the region $\Omega = [0, 40] \times [0, 30]$ m. It is bounded by walls and contains several rectangular shop blocks that people cannot walk through. There are **two exits of equal width $w$**, in the middle of the west wall and the middle of the east wall. A person counts as evacuated, and is removed, once they are 0.5 m outside an exit line.

### 2.2 Assumptions

1. **2D single floor; people are discs** of radius $r_i$, the standard Social Force Model representation [1], [2].
2. **Synthetic geometry** representing a medium-sized mall floor, since no real floor plan or evacuation data are available.
3. **No pre-movement time:** everyone starts moving at $t = 0$, so $T_\text{evac}$ is a movement time and the required widths are a lower bound.
4. **Nearest-exit routing:** each person follows the shortest walking route to the nearest exit and does not re-route to a less crowded exit. The layout is symmetric, so the two exits are loaded roughly equally.
5. **Homogeneous adults, normal (non-panic) walking.** Parameters come from [1], [2] and are assumed to transfer to local crowds.
6. **Target time** $T_\text{target} = 180$ s is assumed.

### 2.3 Equations of motion (Social Force Model)

For each pedestrian $i$ with position $\mathbf{x}_i$ (m), velocity $\mathbf{v}_i$ (m s⁻¹), mass $m_i$ (kg) and radius $r_i$ (m):

$$
\frac{d\mathbf{x}_i}{dt} = \mathbf{v}_i, \qquad
m_i \frac{d\mathbf{v}_i}{dt}
= m_i\,\frac{v_i^0\,\mathbf{e}_i^0(\mathbf{x}_i) - \mathbf{v}_i}{\tau}
+ \sum_{j \ne i} \mathbf{f}_{ij}
+ \sum_{W} \mathbf{f}_{iW}
$$

The first term is the **driving force**: each person relaxes towards their desired velocity over time $\tau$. The **pedestrian–pedestrian force** combines soft repulsion, body compression and sliding friction [2]:

$$
\mathbf{f}_{ij} = \Big[ A\, e^{(r_{ij} - d_{ij})/B} + k\, g(r_{ij} - d_{ij}) \Big]\,\mathbf{n}_{ij}
+ \kappa\, g(r_{ij} - d_{ij})\, \Delta v_{ji}^{t}\, \mathbf{t}_{ij}
$$

Here $d_{ij} = \lVert \mathbf{x}_i - \mathbf{x}_j \rVert$, $r_{ij} = r_i + r_j$, $\mathbf{n}_{ij} = (\mathbf{x}_i - \mathbf{x}_j)/d_{ij}$, $\mathbf{t}_{ij} = (-n_{ij,2}, n_{ij,1})$, $\Delta v^t_{ji} = (\mathbf{v}_j - \mathbf{v}_i)\cdot\mathbf{t}_{ij}$, and $g(x) = \max(x, 0)$. The $k$ and $\kappa$ terms therefore act only when the bodies overlap.

The **pedestrian–wall force** uses the closest point on each wall segment $W$ (outer walls and shop-block edges), at distance $d_{iW}$, with normal $\mathbf{n}_{iW}$ and tangent $\mathbf{t}_{iW}$ [2]:

$$
\mathbf{f}_{iW} = \Big[ A\, e^{(r_i - d_{iW})/B} + k\, g(r_i - d_{iW}) \Big]\,\mathbf{n}_{iW}
- \kappa\, g(r_i - d_{iW})\, (\mathbf{v}_i \cdot \mathbf{t}_{iW})\, \mathbf{t}_{iW}
$$

Speeds are capped at $v_\text{max} = 1.3\,v_i^0$ [1].

### 2.4 Desired direction from a shortest-distance map

$D(\mathbf{x})$ is the shortest walking distance from $\mathbf{x}$ to the nearest exit. It approximately solves $\lVert \nabla D \rVert = 1$ with $D = 0$ at the exits and obstacles impassable. It is computed by a multi-source Dijkstra search [6] on a grid with spacing $h = 0.1$ m, with obstacles inflated by the maximum body radius. The desired direction is

$$
\mathbf{e}_i^0 = -\frac{\nabla D(\mathbf{x}_i)}{\lVert \nabla D(\mathbf{x}_i) \rVert},
$$

so people route around shop blocks instead of walking into them.

### 2.5 Input model (random variates)

| Input | Distribution | Method | Source |
|---|---|---|---|
| Desired speed $v_i^0$ | $\mathcal{N}(1.34, 0.26^2)$ m/s, truncated to [0.56, 2.12] m/s (mean ± 3 s.d.) | rejection sampling | [1] |
| Body radius $r_i$ | $\mathcal{U}(0.25, 0.35)$ m | inverse transform | [2] |
| Start position | uniform on the free floor, no overlaps | random sequential placement with rejection | assumed |
| Start velocity | $\mathbf{0}$ | — | assumed |

### 2.6 Parameters

| Symbol | Meaning | Units | Value | Source |
|---|---|---|---|---|
| $m_i$ | mass | kg | 80 | [2] |
| $\tau$ | relaxation time | s | 0.5 | [2] |
| $A$ | repulsion strength | N | $2 \times 10^3$ | [2] |
| $B$ | repulsion range | m | 0.08 | [2] |
| $k$ | body-compression coefficient | kg s⁻² | $1.2 \times 10^5$ | [2] |
| $\kappa$ | sliding-friction coefficient | kg m⁻¹ s⁻¹ | $2.4 \times 10^5$ | [2] |
| $r_c$ | interaction cut-off, $2r_\text{max} + B\ln(A/1\,\text{N})$ | m | 1.3 | derived |
| $\Delta t$ | time step | s | 0.01 (checked in Section 5) | derived |
| $T_\text{max}$ | simulation time limit | s | 600 | assumed |
| $T_\text{target}$ | evacuation target | s | 180 | assumed |

### 2.7 Numerical method: Velocity-Verlet with a velocity predictor

$$
\begin{aligned}
\mathbf{x}_{n+1} &= \mathbf{x}_n + \mathbf{v}_n\,\Delta t + \tfrac12\,\mathbf{a}_n\,\Delta t^2 \\
\tilde{\mathbf{v}} &= \mathbf{v}_n + \mathbf{a}_n\,\Delta t \\
\mathbf{a}_{n+1} &= \mathbf{F}(\mathbf{x}_{n+1}, \tilde{\mathbf{v}}) / m \\
\mathbf{v}_{n+1} &= \mathbf{v}_n + \tfrac12\left(\mathbf{a}_n + \mathbf{a}_{n+1}\right)\Delta t
\end{aligned}
$$

The driving and friction forces depend on velocity, so plain Velocity-Verlet [3], [4] would be implicit. The Euler predictor $\tilde{\mathbf{v}}$ keeps the scheme explicit while staying **second-order accurate** (for the velocity it is Heun's method). For one person in open space the model reduces to $\dot v = (v_0 - v)/\tau$, with exact solution $v(t) = v_0\left(1 - e^{-t/\tau}\right)$. This is the analytic check in Section 5.

**Stability.** The stiffest time scale is body contact. Two overlapping people oscillate at $\omega = \sqrt{2(k + A/B)/m} \approx 60$ rad s⁻¹, so the step must satisfy $\omega\,\Delta t < 2$, i.e. $\Delta t < 0.033$ s. Sliding friction at a 2 cm overlap needs $\Delta t \lesssim 0.017$ s. The production step is $\Delta t = 0.01$ s.

**Neighbour search.** Forces are truncated at $r_c \approx 1.3$ m, where the repulsion is below 1 N (< 0.5 % of the driving force). Neighbours are found with **cell lists** of side $\ge r_c$ [5], which reduces the cost per step from $O(N^2)$ to $O(N)$.

### 2.8 Outputs and statistics

For each run: the evacuation time $T_\text{evac}$ (last person out), $T_{90}$ (90 % out), the cumulative count $N_\text{out}(t)$, the specific exit flow $J_s = \Delta N_\text{out}/(w\,\Delta t)$ (persons m⁻¹ s⁻¹), and the peak density in front of each exit. Each scenario $(N, w)$ is repeated with $R$ independent seeds. The results are reported as

$$
\overline{T} \pm t_{0.975,\,R-1}\,\frac{s}{\sqrt{R}}
$$

(95 % CI) [9]. The required width $w^*(N)$ is the smallest $w$ whose **upper** 95 % limit is $\le T_\text{target}$, interpolated between the simulated widths. The scaling exponent $\alpha$ is fitted from $\log w^* = \log c + \alpha \log N$.

### 2.9 Concepts used

| Concept | Where it appears |
|---|---|
| **Particle-based simulation** | pedestrians as interacting particles with pair forces; cell-list neighbour search (2.3, 2.7) |
| **Numerical ODE solvers** | Newton's equations integrated with Velocity-Verlet / Heun; stability limit and convergence order (2.7, Section 5) |
| **Input modelling and random variate generation** | truncated-normal desired speeds, uniform radii, random non-overlapping start positions (2.5) |
| **Monte Carlo and the law of large numbers** | replications over seeds and 95 % confidence intervals on every result (2.8, Section 6) |

## 3. Implementation

In [ ]:
def simulate(n, rng):
    """REPLACE with your model. Returns the estimate and its standard error."""
    x, y = rng.random(n), rng.random(n)
    p = np.mean(x**2 + y**2 <= 1.0)
    estimate = 4 * p
    std_err = 4 * np.sqrt(p * (1 - p) / n)
    return estimate, std_err

est, se = simulate(50_000, np.random.default_rng(SEED))
print(f"estimate = {est:.4f} ± {1.96*se:.4f} (95% CI)")

## 4. Verification

At least one check, shown with an `assert` or a tolerance test so a reader can see it pass: an analytic or closed-form result, a limiting case, a conservation law, a stability limit, or recovery of known parameters. A second independent check with a measured error-scaling plot makes the verification more rigorous.

In [ ]:
# Check 1: agreement with the analytic value within the 95% confidence interval
est, se = simulate(200_000, np.random.default_rng(SEED))
assert abs(est - np.pi) <= 1.96 * se, "estimate outside 95% CI of the analytic value"
print(f"PASS: |{est:.4f} - pi| = {abs(est-np.pi):.4f} <= {1.96*se:.4f}")

# Check 2 (error scaling): Monte Carlo error should fall like N^(-1/2)
Ns = np.logspace(2, 6, 9).astype(int)
errs = [np.mean([abs(simulate(n, np.random.default_rng(s))[0] - np.pi) for s in range(20)]) for n in Ns]
slope = np.polyfit(np.log(Ns), np.log(errs), 1)[0]
print(f"fitted slope = {slope:.2f} (expected about -0.5)")
assert -0.65 < slope < -0.35

plt.loglog(Ns, errs, "o-", label="mean |error|")
plt.loglog(Ns, errs[0] * (Ns / Ns[0])**-0.5, "--", label="N^(-1/2)")
plt.xlabel("N"); plt.ylabel("error"); plt.legend(); plt.title("Convergence check"); plt.show()

## 5. Experiments and results

[Parameter study or scenario comparison that answers your question. Report replications/seeds and confidence intervals, not single runs.]

In [ ]:
# REPLACE: your parameter sweep or scenario comparison

## 6. Discussion and limitations

[What the results mean for the real-world question; the most important assumptions and their effect; what you would do next.]

**Web demo reference value:** with default settings the web demo should reproduce [value ± tolerance] from section [N].

## 7. References

IEEE style. The numbers match the citations in Section 2. The same list, with a note on what each source is used for, is in `REFERENCES.md`.

1. D. Helbing and P. Molnár, "Social force model for pedestrian dynamics," *Phys. Rev. E*, vol. 51, no. 5, pp. 4282–4286, 1995, doi: [10.1103/PhysRevE.51.4282](https://doi.org/10.1103/PhysRevE.51.4282).
2. D. Helbing, I. Farkas and T. Vicsek, "Simulating dynamical features of escape panic," *Nature*, vol. 407, pp. 487–490, 2000, doi: [10.1038/35035023](https://doi.org/10.1038/35035023).
3. L. Verlet, "Computer 'experiments' on classical fluids. I. Thermodynamical properties of Lennard-Jones molecules," *Phys. Rev.*, vol. 159, no. 1, pp. 98–103, 1967, doi: [10.1103/PhysRev.159.98](https://doi.org/10.1103/PhysRev.159.98).
4. W. C. Swope, H. C. Andersen, P. H. Berens and K. R. Wilson, "A computer simulation method for the calculation of equilibrium constants for the formation of physical clusters of molecules: Application to small water clusters," *J. Chem. Phys.*, vol. 76, no. 1, pp. 637–649, 1982, doi: [10.1063/1.442716](https://doi.org/10.1063/1.442716).
5. M. P. Allen and D. J. Tildesley, *Computer Simulation of Liquids*, 2nd ed. Oxford, U.K.: Oxford Univ. Press, 2017.
6. E. W. Dijkstra, "A note on two problems in connexion with graphs," *Numer. Math.*, vol. 1, pp. 269–271, 1959, doi: [10.1007/BF01386390](https://doi.org/10.1007/BF01386390).
7. A. Seyfried, O. Passon, B. Steffen, M. Boltes, T. Rupprecht and W. Klingsch, "New insights into pedestrian flow through bottlenecks," *Transp. Sci.*, vol. 43, no. 3, pp. 395–406, 2009, doi: [10.1287/trsc.1090.0263](https://doi.org/10.1287/trsc.1090.0263).
8. T. Kretz, A. Grünebohm and M. Schreckenberg, "Experimental study of pedestrian flow through a bottleneck," *J. Stat. Mech.*, vol. 2006, no. 10, Art. no. P10014, 2006, doi: [10.1088/1742-5468/2006/10/P10014](https://doi.org/10.1088/1742-5468/2006/10/P10014).
9. A. M. Law, *Simulation Modeling and Analysis*, 5th ed. New York, NY, USA: McGraw-Hill, 2015.